# 🧾 Invoice & Receipt Data Extractor
## Day 1 — Environment Setup, CORD Dataset, Baseline PaddleOCR

**Goal:** Run baseline PaddleOCR on 10 CORD invoices and catalog failure modes.

**Pipeline overview:**
```
Invoice Image / PDF
      ↓
PaddleOCR (text detection + recognition)
      ↓
Raw OCR Output (bounding boxes + text)
      ↓
Rule-based field extraction (vendor, date, totals, GST, line items)
      ↓
Structured JSON
```

---

## 📦 Cell 1 — Install Dependencies

In [ ]:
# ── Install PaddlePaddle (GPU build) ──────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
    else:
        print(result.stdout[-500:] or "✓ done")

print("Installing PaddlePaddle GPU...")
run("pip install paddlepaddle-gpu==2.6.1 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q")

print("Installing PaddleOCR...")
run("pip install paddleocr==2.7.3 -q")

print("Installing supporting libraries...")
run("pip install datasets Pillow opencv-python-headless pdf2image matplotlib seaborn tabulate -q")
run("apt-get install -y poppler-utils -q")  # needed by pdf2image

print("\n✅ All packages installed")

## 🔧 Cell 2 — Imports & Helpers

In [ ]:
import os, json, re, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from tabulate import tabulate

warnings.filterwarnings('ignore')

# ── Directory layout ──────────────────────────────────────────────────────────
BASE_DIR    = Path("/content/invoice_ocr")
DATA_DIR    = BASE_DIR / "data" / "cord_samples"
OUTPUT_DIR  = BASE_DIR / "outputs"
VIZ_DIR     = BASE_DIR / "visualizations"

for d in [DATA_DIR, OUTPUT_DIR, VIZ_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {BASE_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Viz dir      : {VIZ_DIR}")
print("✅ Directories ready")

## 📂 Cell 3 — Download CORD Dataset (10 Samples)

In [ ]:
from datasets import load_dataset

print("Loading CORD-v2 from HuggingFace Hub (train split, streaming)...")
print("CORD = Consolidated Receipt Dataset — Korean restaurant receipts, widely used for OCR benchmarking")
print("-" * 70)

# Stream so we don't download the whole ~500 MB dataset
cord = load_dataset(
    "naver-clova-ix/cord-v2",
    split="train",
    streaming=True,
    trust_remote_code=True
)

N_SAMPLES = 10
samples   = []
image_paths = []

for i, sample in enumerate(cord):
    if i >= N_SAMPLES:
        break

    img      = sample["image"]          # PIL Image
    gt_json  = json.loads(sample["ground_truth"])  # dict with parsed fields
    img_path = DATA_DIR / f"receipt_{i:02d}.jpg"

    img.save(img_path, "JPEG", quality=95)
    samples.append({"id": i, "path": str(img_path), "gt": gt_json})
    image_paths.append(img_path)

    w, h = img.size
    print(f"  [{i:02d}] {img_path.name}  |  {w}×{h} px")

print(f"\n✅ Saved {N_SAMPLES} CORD receipts to {DATA_DIR}")

## 👀 Cell 4 — Quick Visual Inspection of Downloaded Receipts

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
fig.suptitle("CORD Dataset — 10 Sample Receipts", fontsize=16, fontweight="bold", y=1.01)

for ax, s in zip(axes.flat, samples):
    img = Image.open(s["path"])
    ax.imshow(img)
    ax.set_title(f"Receipt #{s['id']:02d}\n{img.size[0]}×{img.size[1]}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(VIZ_DIR / "01_sample_receipts.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved → visualizations/01_sample_receipts.png")

## 🔍 Cell 5 — Explore CORD Ground-Truth JSON Structure

In [ ]:
print("=" * 70)
print("CORD Ground-Truth JSON — Receipt #00")
print("=" * 70)

gt = samples[0]["gt"]
print(json.dumps(gt, indent=2, ensure_ascii=False)[:3000])

print("\n" + "=" * 70)
print("Top-level keys across all 10 samples:")
all_keys = set()
for s in samples:
    all_keys.update(s["gt"].keys())
print(sorted(all_keys))

print("\n📌 CORD schema reference:")
print("  gt_parse → structured parsed fields (menu items, sub_total, total, etc.)")
print("  gt_parse.menu       → line items [{nm, cnt, price}, ...]")
print("  gt_parse.sub_total  → subtotals (discount, service charge, etc.)")
print("  gt_parse.total      → final totals (total_price, cashprice, etc.)")

## 🤖 Cell 6 — Initialize PaddleOCR

In [ ]:
from paddleocr import PaddleOCR

print("Initializing PaddleOCR (English + Korean, GPU)...")
print("First run downloads ~100 MB of model weights — please wait.")
print("-" * 70)

ocr_engine = PaddleOCR(
    use_angle_cls=True,    # auto-correct rotated text
    lang="en",             # use 'ch' for Chinese/mixed, 'korean' for KR
    use_gpu=True,
    show_log=False,
    det_db_thresh=0.3,     # detection binarization threshold
    det_db_box_thresh=0.5, # box score threshold
    rec_batch_num=8,       # recognition batch size
)

print("\n✅ PaddleOCR ready")
print("Components loaded:")
print("  • Text Detector  : DB (Differentiable Binarization)")
print("  • Angle Classifier: lightweight CNN for 0°/180° rotation")
print("  • Text Recognizer : SVTR (Scene Text Recognition Transformer)")

## ⚙️ Cell 7 — Core OCR Runner & Field Extractor

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Regex patterns for field extraction
# ──────────────────────────────────────────────────────────────────────────────

PATTERNS = {
    # Dates: 2024-01-15 | 01/15/2024 | 15 Jan 2024 | Jan 15, 2024
    "date": re.compile(
        r"(\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4})"
        r"|(\d{4}[-/.]\d{1,2}[-/.]\d{1,2})"
        r"|((Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4})",
        re.IGNORECASE
    ),
    # GST / GSTIN (India 15-char) or generic tax ID
    "gst_number": re.compile(
        r"(GSTIN?\s*:?\s*([0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}))"
        r"|(GST\s*No\.?\s*:?\s*([A-Z0-9]{10,15}))",
        re.IGNORECASE
    ),
    # Total / grand total / amount due
    "total": re.compile(
        r"(TOTAL|GRAND\s*TOTAL|AMOUNT\s*DUE|BALANCE\s*DUE|NET\s*AMOUNT)\s*[:\-]?\s*([\$₹€£]?\s*[\d,]+\.?\d{0,2})",
        re.IGNORECASE
    ),
    # Tax / GST / VAT amounts
    "tax": re.compile(
        r"(TAX|GST|VAT|SGST|CGST|IGST)\s*[@(]?\s*[\d.]+%?[)\s]*[:\-]?\s*([\$₹€£]?\s*[\d,]+\.?\d{0,2})",
        re.IGNORECASE
    ),
    # Subtotal
    "subtotal": re.compile(
        r"(SUB\s*TOTAL|SUBTOTAL|NET\s*SALES|BEFORE\s*TAX)\s*[:\-]?\s*([\$₹€£]?\s*[\d,]+\.?\d{0,2})",
        re.IGNORECASE
    ),
    # Money amounts on a line (for line-item parsing)
    "money": re.compile(r"[\$₹€£]?\s?([\d,]+\.\d{2})"),
    # Invoice / receipt number
    "invoice_no": re.compile(
        r"(INVOICE\s*#?\s*:?\s*(\w+[-/]?\w+))"
        r"|(RECEIPT\s*#?\s*:?\s*(\w+))"
        r"|(ORDER\s*#?\s*:?\s*(\w+))",
        re.IGNORECASE
    ),
}


# ──────────────────────────────────────────────────────────────────────────────
# run_ocr: run PaddleOCR on one image, return list of (bbox, text, confidence)
# ──────────────────────────────────────────────────────────────────────────────

def run_ocr(image_path: Path) -> list:
    """Returns list of dicts: {bbox, text, confidence}"""
    result = ocr_engine.ocr(str(image_path), cls=True)
    if not result or result[0] is None:
        return []

    lines = []
    for box, (text, conf) in result[0]:
        lines.append({
            "bbox": box,          # [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
            "text": text.strip(),
            "confidence": round(float(conf), 4)
        })
    return lines


# ──────────────────────────────────────────────────────────────────────────────
# extract_fields: heuristic field extraction from OCR lines
# ──────────────────────────────────────────────────────────────────────────────

def extract_fields(ocr_lines: list) -> dict:
    """Best-effort field extraction from raw OCR text lines."""

    full_text  = "\n".join(l["text"] for l in ocr_lines)
    texts      = [l["text"] for l in ocr_lines]

    # --- vendor: assume top 3 lines before any date/amount ----
    vendor_candidates = []
    for t in texts[:6]:
        t_clean = t.strip()
        if len(t_clean) > 3 and not re.search(r"\d{4}|\$|₹|:\s*\d", t_clean):
            vendor_candidates.append(t_clean)
        if len(vendor_candidates) == 2:
            break
    vendor = " ".join(vendor_candidates) if vendor_candidates else "UNKNOWN"

    # --- date ------------------------------------------------
    date = None
    m = PATTERNS["date"].search(full_text)
    if m:
        date = m.group(0).strip()

    # --- gst number ------------------------------------------
    gst_number = None
    m = PATTERNS["gst_number"].search(full_text)
    if m:
        gst_number = m.group(0).strip()

    # --- invoice number --------------------------------------
    invoice_no = None
    m = PATTERNS["invoice_no"].search(full_text)
    if m:
        invoice_no = m.group(0).strip()

    # --- totals ----------------------------------------------
    total    = _extract_amount(PATTERNS["total"],    full_text)
    tax      = _extract_amount(PATTERNS["tax"],      full_text)
    subtotal = _extract_amount(PATTERNS["subtotal"], full_text)

    # --- line items (simple heuristic: lines with text + amount) ----
    line_items = _extract_line_items(texts)

    return {
        "vendor":     vendor,
        "date":       date,
        "invoice_no": invoice_no,
        "gst_number": gst_number,
        "line_items": line_items,
        "subtotal":   subtotal,
        "tax":        tax,
        "total":      total,
    }


def _extract_amount(pattern, text):
    m = pattern.search(text)
    if not m:
        return None
    # last group is the dollar amount
    for g in reversed(m.groups()):
        if g and re.search(r"[\d,]+", g):
            return g.strip()
    return None


def _extract_line_items(texts: list) -> list:
    """Extract lines that look like: [description] [qty?] [price]"""
    items = []
    # Skip first few (vendor) and last few (totals)
    candidates = texts[3:-4] if len(texts) > 10 else texts

    for t in candidates:
        amounts = PATTERNS["money"].findall(t)
        if amounts:
            # Description = everything before the first currency match
            desc = re.split(r"[\$₹€£]\s*[\d,]+|\s{2,}\d", t)[0].strip()
            price = amounts[-1].replace(",", "")  # last amount = price
            if desc and len(desc) > 1:
                items.append({"description": desc, "amount": price})
    return items


print("✅ OCR runner and field extractor defined")

## 🚀 Cell 8 — Run Baseline on All 10 Receipts

In [ ]:
results = []

print(f"{'ID':>3}  {'OCR Lines':>9}  {'Time(s)':>7}  {'Conf Avg':>8}  {'Extracted Fields'}")
print("-" * 70)

for s in samples:
    t0       = time.time()
    ocr_out  = run_ocr(Path(s["path"]))
    elapsed  = time.time() - t0

    fields   = extract_fields(ocr_out)

    avg_conf = (
        sum(l["confidence"] for l in ocr_out) / len(ocr_out)
        if ocr_out else 0.0
    )

    # Count how many of 6 key fields were found
    key_fields = ["vendor", "date", "invoice_no", "gst_number", "total", "line_items"]
    found = sum(1 for k in key_fields if fields.get(k) not in [None, [], "UNKNOWN"])
    found_str = ", ".join(
        k for k in key_fields if fields.get(k) not in [None, [], "UNKNOWN"]
    )

    print(f"{s['id']:>3}  {len(ocr_out):>9}  {elapsed:>7.2f}  {avg_conf:>8.3f}  {found_str}")

    # Save per-receipt result
    out = {
        "receipt_id":   s["id"],
        "image_path":   s["path"],
        "ocr_lines":    ocr_out,
        "extracted":    fields,
        "ground_truth": s["gt"],
        "meta": {
            "n_ocr_lines": len(ocr_out),
            "avg_confidence": round(avg_conf, 4),
            "elapsed_sec": round(elapsed, 3),
        }
    }
    results.append(out)
    with open(OUTPUT_DIR / f"receipt_{s['id']:02d}_result.json", "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

print("-" * 70)
total_time = sum(r["meta"]["elapsed_sec"] for r in results)
print(f"\n✅ Processed 10 receipts in {total_time:.1f}s  ({total_time/10:.2f}s avg per receipt)")

## 📊 Cell 9 — Visualize OCR Bounding Boxes on 4 Receipts

In [ ]:
def draw_ocr_boxes(result, ax, max_boxes=None):
    img = Image.open(result["image_path"])
    ax.imshow(img)
    ax.set_title(f"Receipt #{result['receipt_id']:02d} — {result['meta']['n_ocr_lines']} boxes", fontsize=9)

    lines = result["ocr_lines"]
    if max_boxes:
        lines = lines[:max_boxes]

    for line in lines:
        pts = np.array(line["bbox"], dtype=np.float32)
        conf = line["confidence"]
        color = "#00ff88" if conf >= 0.9 else ("#ffbb00" if conf >= 0.7 else "#ff4444")

        poly = plt.Polygon(pts, fill=False, edgecolor=color, linewidth=0.8, alpha=0.85)
        ax.add_patch(poly)

        # Label with confidence
        cx, cy = pts[:, 0].mean(), pts[:, 1].min() - 3
        ax.text(cx, cy, f"{conf:.2f}", fontsize=4, color=color,
                ha="center", va="bottom", fontweight="bold")

    ax.axis("off")


fig, axes = plt.subplots(2, 2, figsize=(16, 20))
fig.suptitle("PaddleOCR — Detected Text Boxes\n(🟢 conf≥0.9  🟡 conf≥0.7  🔴 conf<0.7)",
             fontsize=13, fontweight="bold")

for ax, r in zip(axes.flat, results[:4]):
    draw_ocr_boxes(r, ax)

plt.tight_layout()
plt.savefig(VIZ_DIR / "02_ocr_bboxes.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → visualizations/02_ocr_bboxes.png")

## 📋 Cell 10 — Print Structured JSON for Receipt #0

In [ ]:
r = results[0]

print("=" * 60)
print("BASELINE EXTRACTED JSON — Receipt #00")
print("=" * 60)
print(json.dumps(r["extracted"], indent=2, ensure_ascii=False))

print("\n" + "=" * 60)
print("CORD GROUND TRUTH — Receipt #00 (gt_parse)")
print("=" * 60)
gt_parse = r["ground_truth"].get("gt_parse", {})
print(json.dumps(gt_parse, indent=2, ensure_ascii=False)[:2000])

## 📈 Cell 11 — Confidence Score Distribution

In [ ]:
all_confs = [l["confidence"] for r in results for l in r["ocr_lines"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PaddleOCR Confidence Score Analysis (10 Receipts)", fontsize=13, fontweight="bold")

# Histogram
axes[0].hist(all_confs, bins=40, color="#4F8EF7", edgecolor="white", alpha=0.85)
axes[0].axvline(0.7, color="#ffbb00", linewidth=2, linestyle="--", label="threshold=0.70")
axes[0].axvline(0.9, color="#00cc66", linewidth=2, linestyle="--", label="threshold=0.90")
axes[0].set_xlabel("Confidence"); axes[0].set_ylabel("# Text Boxes")
axes[0].set_title("Confidence Distribution")
axes[0].legend()

# Per-receipt avg
ids   = [r["receipt_id"] for r in results]
avgs  = [r["meta"]["avg_confidence"] for r in results]
colors = ["#00cc66" if a >= 0.9 else ("#ffbb00" if a >= 0.7 else "#ff4444") for a in avgs]
axes[1].bar(ids, avgs, color=colors, edgecolor="white")
axes[1].axhline(0.85, color="gray", linewidth=1.5, linestyle="--", label="target avg=0.85")
axes[1].set_xlabel("Receipt ID"); axes[1].set_ylabel("Avg Confidence")
axes[1].set_title("Per-Receipt Average Confidence")
axes[1].set_ylim(0, 1.05); axes[1].legend()
for i, (x, y) in enumerate(zip(ids, avgs)):
    axes[1].text(x, y + 0.01, f"{y:.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(VIZ_DIR / "03_confidence_analysis.png", dpi=120, bbox_inches="tight")
plt.show()

low = sum(1 for c in all_confs if c < 0.7)
print(f"\nTotal text boxes : {len(all_confs)}")
print(f"High conf (≥0.9) : {sum(1 for c in all_confs if c>=0.9)} ({100*sum(1 for c in all_confs if c>=0.9)/len(all_confs):.1f}%)")
print(f"Medium (0.7-0.9) : {sum(1 for c in all_confs if 0.7<=c<0.9)} ({100*sum(1 for c in all_confs if 0.7<=c<0.9)/len(all_confs):.1f}%)")
print(f"Low conf (<0.7)  : {low} ({100*low/len(all_confs):.1f}%)  ← FAILURE ZONE")

## 🔬 Cell 12 — Field Extraction Summary Table

In [ ]:
FIELDS = ["vendor", "date", "invoice_no", "gst_number", "subtotal", "tax", "total"]

rows = []
for r in results:
    ext = r["extracted"]
    row = [r["receipt_id"]]
    for f in FIELDS:
        val = ext.get(f)
        if val in [None, [], "UNKNOWN"]:
            row.append("✗")
        elif f == "line_items":
            row.append(f"✓ ({len(val)} items)")
        else:
            short = str(val)[:18] + "…" if len(str(val)) > 18 else str(val)
            row.append(f"✓ {short}")
    rows.append(row)

headers = ["Receipt"] + FIELDS
print(tabulate(rows, headers=headers, tablefmt="grid"))

# Success rate per field
print("\n📊 Field extraction success rate:")
for f in FIELDS:
    found = sum(
        1 for r in results
        if r["extracted"].get(f) not in [None, [], "UNKNOWN"]
    )
    bar = "█" * found + "░" * (10 - found)
    print(f"  {f:<12}: [{bar}] {found}/10  ({found*10}%)")

## 🐛 Cell 13 — Systematic Failure Mode Analysis

In [ ]:
print("=" * 70)
print("FAILURE MODE ANALYSIS — PaddleOCR Baseline")
print("=" * 70)

# ── Failure Mode Detection Heuristics ─────────────────────────────────────────

failure_catalog = []

for r in results:
    rid     = r["receipt_id"]
    lines   = r["ocr_lines"]
    ext     = r["extracted"]
    texts   = [l["text"] for l in lines]
    confs   = [l["confidence"] for l in lines]

    failures = []

    # FM-1: No / very few OCR lines → complete detection failure
    if len(lines) < 5:
        failures.append(("FM-1", "COMPLETE OCR FAILURE",
                         f"Only {len(lines)} lines detected"))

    # FM-2: Very low average confidence → degraded image quality
    avg_conf = sum(confs)/len(confs) if confs else 0
    if 0 < avg_conf < 0.70:
        failures.append(("FM-2", "LOW CONFIDENCE",
                         f"Avg conf={avg_conf:.2f} — blurry/dark image?"))

    # FM-3: Non-ASCII / garbled text (likely Korean mis-read as English)
    garbled = [t for t in texts if re.search(r"[\x80-\xff]", t)]
    non_ascii_ratio = len(garbled) / len(texts) if texts else 0
    if non_ascii_ratio > 0.3:
        failures.append(("FM-3", "NON-ASCII / LANGUAGE MISMATCH",
                         f"{len(garbled)}/{len(texts)} lines contain non-ASCII chars"))

    # FM-4: Missed total
    if ext["total"] is None:
        # Check if total-like word even present
        has_total_word = any(re.search(r"total|amount|due", t, re.I) for t in texts)
        if has_total_word:
            failures.append(("FM-4", "TOTAL REGEX MISS",
                             "'total' keyword found but amount not extracted — unusual format"))
        else:
            failures.append(("FM-4", "TOTAL NOT DETECTED",
                             "No total keyword in OCR output"))

    # FM-5: Fragmented line items (description split across multiple boxes)
    if ext["line_items"]:
        short_descs = [i for i in ext["line_items"] if len(i["description"]) < 3]
        if len(short_descs) > 2:
            failures.append(("FM-5", "FRAGMENTED LINE ITEMS",
                             f"{len(short_descs)} line items have description <3 chars"))

    # FM-6: Missing date
    if ext["date"] is None:
        failures.append(("FM-6", "DATE NOT FOUND",
                         "Regex did not match any date pattern"))

    # FM-7: Missing GST (CORD is Korean receipts, expected for baseline)
    if ext["gst_number"] is None:
        failures.append(("FM-7", "GST/TAX ID NOT FOUND",
                         "CORD receipts are Korean → no Indian GSTIN expected (known limitation)"))

    failure_catalog.append({"receipt_id": rid, "failures": failures})

    if failures:
        print(f"\n▶ Receipt #{rid:02d}:")
        for code, name, detail in failures:
            print(f"    [{code}] {name}")
            print(f"           {detail}")
    else:
        print(f"\n▶ Receipt #{rid:02d}: ✓ No critical failures detected")

print("\n" + "=" * 70)

## 📉 Cell 14 — Failure Mode Summary Chart

In [ ]:
from collections import Counter

# Count occurrences of each failure code
fm_counter = Counter()
fm_names   = {}
for entry in failure_catalog:
    for code, name, _ in entry["failures"]:
        fm_counter[code] += 1
        fm_names[code] = name

if fm_counter:
    codes  = sorted(fm_counter.keys())
    counts = [fm_counter[c] for c in codes]
    labels = [f"{c}\n{fm_names[c][:20]}" for c in codes]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(labels, counts, color=sns.color_palette("Reds_r", len(codes)),
                  edgecolor="white", zorder=3)
    ax.set_title("Failure Mode Frequency Across 10 Baseline Receipts",
                 fontsize=13, fontweight="bold")
    ax.set_ylabel("# Receipts Affected")
    ax.set_ylim(0, 12)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                str(cnt), ha="center", fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.savefig(VIZ_DIR / "04_failure_modes.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved → visualizations/04_failure_modes.png")
else:
    print("No failures detected — excellent baseline!")

## 📝 Cell 15 — Day 1 Failure Mode Report

In [ ]:
report = {
    "day": 1,
    "timestamp": datetime.now().isoformat(),
    "dataset": "CORD-v2 (naver-clova-ix/cord-v2), 10 train samples",
    "model": "PaddleOCR 2.7.3 — DB detector + SVTR recognizer, lang=en",
    "summary": {
        "total_receipts": 10,
        "avg_ocr_lines":  round(sum(r["meta"]["n_ocr_lines"] for r in results)/10, 1),
        "avg_confidence": round(sum(r["meta"]["avg_confidence"] for r in results)/10, 3),
        "avg_latency_sec":round(sum(r["meta"]["elapsed_sec"] for r in results)/10, 3),
    },
    "field_extraction_rate": {},
    "failure_modes": {
        "FM-1": {"name": "Complete OCR failure",
                 "root_cause": "Image too dark, low-res, or extreme skew",
                 "fix_day2":   "Pre-process: adaptive thresholding + deskew (OpenCV)"},
        "FM-2": {"name": "Low confidence scores",
                 "root_cause": "Blur, noise, unusual fonts, watermarks",
                 "fix_day2":   "CLAHE contrast enhancement, denoising, DPI upscaling"},
        "FM-3": {"name": "Language / charset mismatch",
                 "root_cause": "CORD = Korean receipts; model=en misreads Hangul",
                 "fix_day2":   "Use lang=korean or multilingual model; auto-detect language"},
        "FM-4": {"name": "Total field not extracted",
                 "root_cause": "Non-standard total labels, OCR merge errors on amount lines",
                 "fix_day3":   "Train NER / LayoutLM to recognize field semantics from position"},
        "FM-5": {"name": "Fragmented line items",
                 "root_cause": "Multi-column receipt layout; OCR boxes don't span full line",
                 "fix_day3":   "Spatial grouping: cluster boxes by y-coordinate, merge by row"},
        "FM-6": {"name": "Date not found",
                 "root_cause": "Non-standard date formats (e.g. '25 JAN 24' or locale-specific)",
                 "fix_day2":   "Extend regex; add dateparser library as fallback"},
        "FM-7": {"name": "GST/Tax ID not found",
                 "root_cause": "CORD (Korean receipts) don't have Indian GSTIN numbers",
                 "fix_note":   "Expected on real Indian invoices; add BRN/TIN patterns for other regions"},
    },
    "next_steps": [
        "Day 2: Image pre-processing pipeline (deskew, denoise, binarize)",
        "Day 2: Switch to multilingual/Korean OCR model for CORD; test on real Indian invoices",
        "Day 3: Spatial line grouping — merge OCR boxes by row proximity",
        "Day 3: Replace regex with LayoutLMv3 for semantic field detection",
        "Day 4: PDF ingestion (pdf2image + pdfplumber fallback for digital PDFs)",
        "Day 5: Claude API integration for LLM-based structured JSON extraction",
    ]
}

# Fill field rates
for f in FIELDS:
    found = sum(1 for r in results if r["extracted"].get(f) not in [None, [], "UNKNOWN"])
    report["field_extraction_rate"][f] = f"{found}/10 ({found*10}%)"

# Save report
report_path = OUTPUT_DIR / "day1_baseline_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(f"\n✅ Report saved → {report_path}")

## 💾 Cell 16 — Package All Outputs

In [ ]:
import zipfile

zip_path = "/content/day1_invoice_ocr_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.glob("*.json"):
        zf.write(f, f"outputs/{f.name}")
    for f in VIZ_DIR.glob("*.png"):
        zf.write(f, f"visualizations/{f.name}")

print(f"✅ All outputs zipped → {zip_path}")
print("\nContents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"  {name:<50} {info.file_size/1024:>8.1f} KB")

# Download link
try:
    from google.colab import files
    files.download(zip_path)
    print("\n⬇️  Download started in browser")
except:
    print(f"\nTo download, run in a new cell: from google.colab import files; files.download('{zip_path}')")

---
## ✅ Day 1 Complete

### What we built
| Step | Status |
|------|--------|
| Environment setup (PaddleOCR, datasets, OpenCV) | ✅ |
| CORD-v2 dataset download (10 samples) | ✅ |
| PaddleOCR baseline inference | ✅ |
| Heuristic JSON field extractor (vendor, date, GST, totals, line items) | ✅ |
| Confidence score analysis | ✅ |
| Failure mode catalog (FM-1 through FM-7) | ✅ |

### Failure Modes Observed
| Code | Problem | Impact |
|------|---------|--------|
| **FM-1** | Complete OCR failure on poor quality scans | No data extracted |
| **FM-2** | Low confidence (<0.7) on blurry/dark regions | Wrong text |
| **FM-3** | Language mismatch (Korean read as English) | Garbled output |
| **FM-4** | Total field regex miss (non-standard labels) | Missing totals |
| **FM-5** | Fragmented line items (multi-column layout) | Broken item list |
| **FM-6** | Date regex misses locale-specific formats | Missing date |
| **FM-7** | GST number absent (CORD is Korean) | Expected gap |

### Day 2 Plan
- Image pre-processing pipeline (deskew → denoise → binarize)
- Spatial box grouping to fix fragmented line items
- Multilingual OCR / language auto-detection
- Extended date & tax-ID patterns